In [ ]:
%pip install -q huggingface_hub tqdm pandas matplotlib

In [ ]:
import os, re, json, math, random, hashlib, zipfile, gc, time, pickle, warnings, sys, copy, shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

SMOKE_TEST = False
SEED = 42
RESUME_COMPLETED_RUNS = True
WRITE_SUBMISSIONS = True

DATASET_ID = "Exploration-Lab/CS-781-Capstone"
DATA_SOURCE = "hf"
LOCAL_DATA_DIR = None
SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working", "."]
PHASE1_CKPT_OVERRIDE = None
TOKENIZER_PATH_OVERRIDE = None

CONTEXT = "document"

CHUNK_SENTS = 32
CHUNK_STRIDE = 16
MAX_SENT_TOKENS = 128
WINDOW = 512
XL_MEMORY = 0
XL_MEMORY_OLD = 0
POOLING = "mean+last"
SENT_DIM = 512
LSTM_HIDDEN = 256
DOMAIN_EMB_DIM = 16
CRF_AUX_CE_WEIGHT = 0.5
CLASS_WEIGHT = "sqrt_inv"
RARE_OVERSAMPLE_CAP = 3.0

BATCH_CHUNKS = 2
GRAD_ACCUM = 4
EVAL_BATCH_CHUNKS = 4
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06
GRAD_CLIP = 1.0
HEAD_DROPOUT = 0.1
BACKBONE_DROPOUT = 0.1
USE_GRAD_CHECKPOINTING = False

EXPERIMENT_LIBRARY = {
    "lorma_plus": dict(method="lorma_plus", rank=16, alpha=16, lr=5e-4, head_lr=1e-3, epochs=20, patience=5,
                       description="LoRMA+ (compulsory): multiplicative low-rank update W=(I+a/r*BA)W0."),
    "lora": dict(method="lora", rank=16, alpha=32, lora_dropout=0.05, lr=1e-3, head_lr=1e-3, epochs=20, patience=5,
                 description="LoRA: additive low-rank update W0 + (a/r) BA."),
    "rslora": dict(method="rslora", rank=32, alpha=16, lora_dropout=0.05, lr=1e-3, head_lr=1e-3, epochs=20, patience=5,
                   description="Rank-stabilised LoRA: scaling a/sqrt(r) instead of a/r (stable at higher rank)."),
    "qlora": dict(method="qlora", rank=16, alpha=32, lora_dropout=0.05, lr=1e-3, head_lr=1e-3, epochs=20, patience=5,
                  description="QLoRA: frozen base weights stored in 4-bit NF4 (block 64, absmax) + LoRA in 16/32-bit."),
    "ia3": dict(method="ia3", lr=3e-3, head_lr=1e-3, epochs=25, patience=6,
                description="IA3: learned vectors rescale keys, values and the FFN inner activations (~0.1% params)."),
    "prefix": dict(method="prefix", prefix_len=32, prefix_hidden=256, lr=1e-3, head_lr=1e-3, epochs=20, patience=5,
                   description="Prefix tuning: trainable per-layer prefix states (MLP re-parameterised) in every layer's memory."),
    "full": dict(method="full", lr=1e-4, head_lr=1e-3, epochs=12, patience=4,
                 description="Full fine-tuning (reference upper bound for the PEFT methods)."),
    "distill": dict(method="distill", kd_temperature=2.0, kd_alpha=0.5, kd_hidden_weight=0.2, teacher_ensemble=1,
                    lr=1e-4, head_lr=1e-3, epochs=15, patience=4,
                    description="Knowledge distillation: fine-tuned teacher -> smaller student (logits + hidden states)."),
}
EXPERIMENT_DEFAULTS = dict(context=None, head=None, batch_chunks=None, grad_accum=None, dropout=None, use_amp=True)

def make_experiment(base, **over):
    cfg = copy.deepcopy(EXPERIMENT_LIBRARY[base])
    cfg.update(over)
    return cfg

ARCH_NAME = "multires_memory"
OUTPUT_NAME = "phase3_multires_memory_rr"
MIN_PARAMS, MAX_PARAMS = 40e6, 100e6
AUTO_TRIM_LAYERS = True
PHASE1_ARCH_KEYWORD = "multires"
XL_MEMORY = 256
XL_MEMORY_OLD = 0
LOWRANK_TARGETS = ("attn.q", "attn.k", "attn.v", "attn.o", "ff1", "ff2")
IA3_TARGETS = {"attn.k": "out", "attn.v": "out", "ff2": "in"}
EXPERIMENTS = {
    "E2_prefix_tuning": make_experiment("prefix"),
    "E3_rslora": make_experiment("rslora"),
    "E4_distill": make_experiment("distill", student_layers=6,
                                  description="KD: best fine-tuned teacher -> 6-layer student from evenly spaced Phase-1 layers (>= 40M)."),
}
RUN_LIST = ["E2_prefix_tuning", "E3_rslora", "E4_distill"]

OUTPUT_DIR = Path("/kaggle/working") / OUTPUT_NAME if Path("/kaggle/working").exists() else Path(OUTPUT_NAME)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "runs").mkdir(exist_ok=True)

if SMOKE_TEST:
    WINDOW, CHUNK_SENTS, CHUNK_STRIDE, MAX_SENT_TOKENS = 64, 8, 4, 24
    BATCH_CHUNKS, GRAD_ACCUM, EVAL_BATCH_CHUNKS, SENT_DIM, LSTM_HIDDEN = 2, 1, 4, 32, 16
    XL_MEMORY = min(XL_MEMORY, 16)
    XL_MEMORY_OLD = min(XL_MEMORY_OLD, 16)
    MIN_PARAMS = 0
    for _cfg in EXPERIMENTS.values():
        _cfg["epochs"] = 2
        _cfg["patience"] = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          "| memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
elif not SMOKE_TEST:
    raise RuntimeError("Enable a GPU accelerator (Kaggle: Settings > Accelerator > GPU T4/P100).")
print("Experiments to run:", RUN_LIST)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

def save_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)

def make_grad_scaler(enabled):
    if not enabled:
        return None
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=True)

In [ ]:
class ByteBPE:
    EOW = 260

    def __init__(self):
        self.specials = ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos, self.stoi, self.payload, self.merges = [], {}, {}, []
        self.vocab_size = 0
        self._ranks = {}
        self._cache = {}

    @classmethod
    def from_json(cls, payload):
        t = cls()
        t.itos = t.specials + [f"<0x{i:02X}>" for i in range(256)] + ["</w>"]
        t.stoi = {s: i for i, s in enumerate(t.itos)}
        t.payload = {i: bytes([i - 4]) for i in range(4, 260)}
        t.payload[260] = b""
        t.merges = [tuple(map(int, p)) for p in payload["merges"]]
        for pair in t.merges:
            new_id = len(t.itos)
            symbol = f"<{pair[0]}+{pair[1]}>"
            t.itos.append(symbol)
            t.stoi[symbol] = new_id
            t.payload[new_id] = t.payload.get(pair[0], b"") + t.payload.get(pair[1], b"")
        t.vocab_size = len(t.itos)
        if t.itos != payload.get("itos"):
            raise ValueError("Saved ByteBPE vocabulary and merge list are inconsistent.")
        t._ranks = {pair: r for r, pair in enumerate(t.merges)}
        return t

    def encode_word_reference(self, word):
        seq = [b + 4 for b in str(word).encode("utf-8")] + [self.EOW]
        for rank, pair in enumerate(self.merges):
            out, j = [], 0
            while j < len(seq):
                if j + 1 < len(seq) and (seq[j], seq[j + 1]) == pair:
                    out.append(261 + rank); j += 2
                else:
                    out.append(seq[j]); j += 1
            seq = out
        return seq

    def encode_piece(self, piece, add_eow=True):
        key = (piece, add_eow)
        hit = self._cache.get(key)
        if hit is not None:
            return hit
        seq = [b + 4 for b in piece.encode("utf-8")]
        if add_eow:
            seq.append(self.EOW)
        ranks = self._ranks
        while len(seq) > 1:
            best_rank, best_pair = None, None
            for i in range(len(seq) - 1):
                r = ranks.get((seq[i], seq[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_pair = r, (seq[i], seq[i + 1])
            if best_pair is None:
                break
            out, j, new_id = [], 0, 261 + best_rank
            while j < len(seq):
                if j + 1 < len(seq) and (seq[j], seq[j + 1]) == best_pair:
                    out.append(new_id); j += 2
                else:
                    out.append(seq[j]); j += 1
            seq = out
        if len(self._cache) < 400_000:
            self._cache[key] = seq
        return seq

def _sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def _search(patterns, roots):
    found, seen = [], set()
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pat in patterns:
            for p in sorted(root.rglob(pat)):
                rp = str(p.resolve())
                if p.is_file() and rp not in seen:
                    seen.add(rp); found.append(p)
    return found

MULTIRES_KEYS = ("d_model", "layers", "query_heads", "kv_heads", "ff_dim", "recent_tokens", "compress_group_size")
if PHASE1_CKPT_OVERRIDE:
    ckpt_candidates = [Path(PHASE1_CKPT_OVERRIDE)]
else:
    ckpt_candidates = _search(["best_multires*.pt", "best*.pt"], SEARCH_ROOTS) + \
                      _search(["latest_multires*.pt", "*.pt", "*.pth"], SEARCH_ROOTS)
_payload, PHASE1_CKPT_PATH = None, None
for cand in ckpt_candidates:
    try:
        pl = torch.load(cand, map_location="cpu", weights_only=False)
    except Exception as exc:
        print(f"  skip {cand}: {exc}"); continue
    cfg_ = (pl.get("config") or pl.get("model_config") or {}) if isinstance(pl, dict) else {}
    if isinstance(pl, dict) and "model_state_dict" in pl and all(k in cfg_ for k in MULTIRES_KEYS):
        _payload, PHASE1_CKPT_PATH = pl, cand
        break
    print(f"  skip {cand}: not a multi-resolution-memory checkpoint "
          f"(architecture={pl.get('architecture', '?') if isinstance(pl, dict) else '?'})")
    del pl
if _payload is None:
    raise FileNotFoundError(
        "No multi-resolution-memory Phase-1 checkpoint found. Attach the Phase-1 output containing "
        "best_multires_gqa_rmsnorm.pt ('Add Input') or set PHASE1_CKPT_OVERRIDE.")
print("Phase-1 checkpoint:", PHASE1_CKPT_PATH)

P1_STATE = _payload["model_state_dict"]
P1_CFG = dict(_payload.get("config") or _payload.get("model_config"))
P1_VOCAB = int(_payload.get("vocab_size", P1_STATE["emb.weight"].shape[0]))
P1_CFG["vocab_size"] = P1_VOCAB
P1_TOKENIZER_SHA = _payload.get("tokenizer_sha256")
P1_ARCH = "multi_resolution_memory_gqa_rmsnorm_rope"
P1_CFG.setdefault("rope_base", 10_000.0)
P1_CFG.setdefault("rms_eps", 1e-6)
print("Phase-1 architecture:", P1_ARCH)
print("Phase-1 config:", {k: P1_CFG.get(k) for k in MULTIRES_KEYS + ("old_history_tokens", "segment_length", "vocab_size")})
del _payload
gc.collect()

if TOKENIZER_PATH_OVERRIDE:
    tok_candidates = [Path(TOKENIZER_PATH_OVERRIDE)]
else:
    tok_candidates = _search(["legal_bpe_tokenizer*.json", "*bpe*.json", "*tokenizer*.json"], SEARCH_ROOTS)

TOKENIZER, TOKENIZER_PATH, TOKENIZER_SHA256 = None, None, None
tok_table = []
for cand in tok_candidates:
    try:
        raw = json.loads(cand.read_text(encoding="utf-8"))
        if not (isinstance(raw, dict) and "merges" in raw and "itos" in raw):
            continue
        sha = _sha256_file(cand)
        tok_table.append((cand, int(raw.get("vocab_size", len(raw["itos"]))), sha, raw))
    except Exception:
        continue

print("Tokenizer files found:")
for cand, vs, sha, _ in tok_table:
    print(f"  {cand}  vocab={vs}  sha256={sha[:12]}")

chosen = [t for t in tok_table if P1_TOKENIZER_SHA and t[2] == P1_TOKENIZER_SHA]
how = "sha256 match with the Phase-1 checkpoint"
if not chosen:
    chosen = [t for t in tok_table if t[1] == P1_VOCAB]
    how = f"vocab size match ({P1_VOCAB}); sha256 differs/unknown - verify this is your Phase-1 tokenizer file"
if not chosen:
    raise FileNotFoundError(
        f"No tokenizer with vocab size {P1_VOCAB} found. Attach your legal_bpe_tokenizer JSON for that size "
        "(or set TOKENIZER_PATH_OVERRIDE). This notebook never trains a tokenizer.")
TOKENIZER_PATH, _, TOKENIZER_SHA256, _raw = chosen[0]
TOKENIZER = ByteBPE.from_json(_raw)
del tok_table, _raw
print(f"Using tokenizer: {TOKENIZER_PATH} ({how})")

PAD_ID, UNK_ID, BOS_ID, EOS_ID = (TOKENIZER.stoi[s] for s in ("<pad>", "<unk>", "<s>", "</s>"))
EOW_ID = ByteBPE.EOW
VOCAB_SIZE = TOKENIZER.vocab_size
assert VOCAB_SIZE == P1_VOCAB, f"Tokenizer vocab {VOCAB_SIZE} != checkpoint vocab {P1_VOCAB}"
print("Vocab size:", VOCAB_SIZE, "| ids:", dict(pad=PAD_ID, unk=UNK_ID, bos=BOS_ID, eos=EOS_ID, eow=EOW_ID))

_rng = random.Random(0)
_words = ["The", "Supreme", "Court", "of", "India,", "Section", "302", "IPC.", "(1995)", "Crl.A.No.123/2005",
          "appellant-State", "Hon'ble", "₹5,000", "naïve"]
for _w in _words:
    assert TOKENIZER.encode_piece(_w, True) == TOKENIZER.encode_word_reference(_w), _w
print("Fast BPE encoder == Phase-1 reference encoder on self-check words.")

In [ ]:
DOMAINS = ["CL", "IT"]
FILE_NAMES = [f"{d}_{s}.json" for d in DOMAINS for s in ("train", "dev", "test")] + ["label_vocab.json"]
DEFAULT_LABELS = ["Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", "Dissent",
                  "PrecedentReliedUpon", "PrecedentNotReliedUpon", "PrecedentOverruled", "RulingByLowerCourt",
                  "RatioOfTheDecision", "RulingByPresentCourt", "None"]

def locate_phase3_files():
    paths = {}
    if DATA_SOURCE == "local":
        roots = [LOCAL_DATA_DIR] if LOCAL_DATA_DIR else SEARCH_ROOTS
        for name in FILE_NAMES:
            hits = _search([name], roots)
            if not hits:
                raise FileNotFoundError(f"{name} not found under {roots}")
            paths[name] = hits[0]
        return paths
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            token = None
    if not token:
        from getpass import getpass
        token = getpass("Hugging Face read token: ")
    from huggingface_hub import HfApi, hf_hub_download
    repo_files = HfApi().list_repo_files(DATASET_ID, repo_type="dataset", token=token)
    for name in FILE_NAMES:
        hits = [f for f in repo_files if f.endswith("/" + name) or f == name]
        pref = [f for f in hits if "phase_3" in f.lower() or "_rr" in f.lower()]
        hits = pref or hits
        if not hits:
            raise FileNotFoundError(f"{name} not found in {DATASET_ID}. Some files: {repo_files[:30]}")
        paths[name] = Path(hf_hub_download(DATASET_ID, hits[0], repo_type="dataset", token=token))
    return paths

P3_PATHS = locate_phase3_files()
for k, v in P3_PATHS.items():
    print(f"{k:18s} -> {v}")

_lv = json.loads(Path(P3_PATHS["label_vocab.json"]).read_text(encoding="utf-8"))
if isinstance(_lv, dict):
    if all(isinstance(v, int) for v in _lv.values()):
        LABELS = [k for k, _ in sorted(_lv.items(), key=lambda kv: kv[1])]
    elif all(isinstance(k, str) and k.isdigit() for k in _lv):
        LABELS = [v for _, v in sorted(_lv.items(), key=lambda kv: int(kv[0]))]
    else:
        LABELS = list(_lv.get("labels", _lv.keys()))
else:
    LABELS = [str(x) for x in _lv]
if not LABELS:
    LABELS = DEFAULT_LABELS
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
print(f"{NUM_LABELS} rhetorical roles:", LABELS)

def _as_doc_list(obj):
    if isinstance(obj, dict):
        for key in ("data", "documents", "docs"):
            if key in obj and isinstance(obj[key], list):
                return obj[key]
        out = []
        for k, v in obj.items():
            if isinstance(v, dict):
                v = dict(v); v.setdefault("id", k); out.append(v)
            elif isinstance(v, list):
                out.append({"id": k, "text": v})
        return out
    return obj

def load_docs(path, domain, with_labels):
    docs = []
    for i, d in enumerate(_as_doc_list(json.loads(Path(path).read_text(encoding="utf-8")))):
        did = str(d.get("id", d.get("doc_id", d.get("document_id", f"{domain}_{i}"))))
        sents = d.get("text", d.get("sentences", d.get("sents")))
        if isinstance(sents, str):
            raise ValueError(f"{did}: expected a list of sentences")
        sents = ["" if s is None else str(s) for s in sents]
        doc = {"id": did, "domain": domain, "sentences": sents}
        if with_labels:
            labs = [str(x) for x in d.get("labels", d.get("label", []))]
            if len(labs) != len(sents):
                print(f"WARNING {did}: {len(sents)} sentences vs {len(labs)} labels - truncating to the shorter")
                n = min(len(sents), len(labs)); doc["sentences"], labs = sents[:n], labs[:n]
            unknown = [l for l in labs if l not in LABEL2ID]
            if unknown:
                raise ValueError(f"{did}: labels not in label_vocab.json: {sorted(set(unknown))[:5]}")
            doc["labels"] = [LABEL2ID[l] for l in labs]
        docs.append(doc)
    return docs

DATA = {"train": [], "dev": [], "test": []}
for dom in DOMAINS:
    for split in ("train", "dev", "test"):
        part = load_docs(P3_PATHS[f"{dom}_{split}.json"], dom, with_labels=(split != "test"))
        DATA[split].extend(part)
        print(f"[{dom} {split:5s}] docs={len(part):4d} sentences={sum(len(d['sentences']) for d in part):6d}")
test_ids = [d["id"] for d in DATA["test"]]
assert len(test_ids) == len(set(test_ids)), "test document ids collide across CL/IT - prediction.json needs unique ids"
DOMAIN2ID = {d: i for i, d in enumerate(DOMAINS)}

import pandas as pd
_rows = {}
for dom in DOMAINS:
    c = Counter(LABELS[l] for d in DATA["train"] if d["domain"] == dom for l in d["labels"])
    _rows[dom] = [c.get(l, 0) for l in LABELS]
LABEL_STATS = pd.DataFrame(_rows, index=LABELS)
LABEL_STATS["total"] = LABEL_STATS.sum(1)
print(LABEL_STATS)
TRAIN_LABEL_COUNTS = np.array(LABEL_STATS["total"].values, dtype=np.float64)

In [ ]:
def encode_sentence(s):
    ids = []
    for w in s.split():
        ids.extend(TOKENIZER.encode_piece(w, True))
        if len(ids) >= MAX_SENT_TOKENS:
            break
    return (ids or [UNK_ID])[:MAX_SENT_TOKENS]

_tok_cache = OUTPUT_DIR / f"p3_tokens_{TOKENIZER_SHA256[:8]}_{MAX_SENT_TOKENS}_{sum(len(v) for v in DATA.values())}.pkl"
if _tok_cache.exists():
    with _tok_cache.open("rb") as f:
        _cached = pickle.load(f)
    for split in DATA:
        for d in DATA[split]:
            d.update(_cached[(split, d["id"])])
    print("Loaded cached sentence tokenisation.")
else:
    _cached = {}
    for split in DATA:
        for d in tqdm(DATA[split], desc=f"tokenising {split}"):
            toks = [np.asarray(encode_sentence(s), dtype=np.int32) for s in d["sentences"]]
            lens = np.array([len(t) for t in toks], dtype=np.int64)
            off = np.concatenate([[0], np.cumsum(lens)[:-1]]) if len(lens) else np.zeros(0, dtype=np.int64)
            info = {"stream": np.concatenate(toks) if toks else np.zeros(0, np.int32), "sent_off": off, "sent_len": lens}
            d.update(info)
            _cached[(split, d["id"])] = info
    with _tok_cache.open("wb") as f:
        pickle.dump(_cached, f, protocol=4)

_sl = np.concatenate([d["sent_len"] for d in DATA["train"]])
print(f"tokens/sentence: median={int(np.median(_sl))} p95={int(np.percentile(_sl, 95))} "
      f"| truncated at {MAX_SENT_TOKENS}: {np.mean(_sl >= MAX_SENT_TOKENS):.1%} of train sentences")
print(f"sentences/document (train): median={int(np.median([len(d['sentences']) for d in DATA['train']]))} "
      f"max={max(len(d['sentences']) for d in DATA['train'])}")

_mx = TRAIN_LABEL_COUNTS.max()
LABEL_OVERSAMPLE = np.array([min(RARE_OVERSAMPLE_CAP, math.sqrt(_mx / c)) if c > 0 else 1.0 for c in TRAIN_LABEL_COUNTS])

def pack_windows(doc, s0, s1, context):
    wins, cur, cur_len = [], [], 0
    for s in range(s0, s1):
        L = int(doc["sent_len"][s])
        if cur and (context == "sentence" or cur_len + L > WINDOW):
            wins.append(cur); cur, cur_len = [], 0
        cur.append(s); cur_len += L
    if cur:
        wins.append(cur)
    return wins

def make_chunk(doc, s0, s1, context, with_labels=True):
    windows = []
    for sents in pack_windows(doc, s0, s1, context):
        a = int(doc["sent_off"][sents[0]])
        b = int(doc["sent_off"][sents[-1]] + doc["sent_len"][sents[-1]])
        mem = doc["stream"][max(0, a - XL_MEMORY - XL_MEMORY_OLD):a] if (context == "document" and XL_MEMORY) else \
            np.zeros(0, np.int32)
        spans = [(s - s0, int(doc["sent_off"][s]) - a, int(doc["sent_off"][s] + doc["sent_len"][s]) - a) for s in sents]
        windows.append(dict(ids=doc["stream"][a:b].astype(np.int64), start=a, mem=mem.astype(np.int64), spans=spans))
    item = dict(windows=windows, n=s1 - s0, s0=s0, domain=DOMAIN2ID[doc["domain"]])
    if with_labels:
        item["labels"] = np.asarray(doc["labels"][s0:s1], dtype=np.int64)
    return item

def chunk_starts_eval(n):
    if n <= CHUNK_SENTS:
        return [0]
    st = list(range(0, n - CHUNK_SENTS + 1, CHUNK_STRIDE))
    if st[-1] + CHUNK_SENTS < n:
        st.append(n - CHUNK_SENTS)
    return st

class TrainChunks(Dataset):
    def __init__(self, docs, context, rng):
        self.docs, self.context, self.items = docs, context, []
        size = CHUNK_SENTS
        for di, d in enumerate(docs):
            n = len(d["sentences"])
            if n == 0:
                continue
            off = rng.randrange(size) if n > size else 0
            bounds = sorted(set([0, n] + ([off] if 0 < off < n else []) + list(range(off + size, n, size))))
            for s0, s1 in zip(bounds[:-1], bounds[1:]):
                labs = np.unique(np.asarray(d["labels"][s0:s1]))
                w = float(LABEL_OVERSAMPLE[labs].max()) if len(labs) else 1.0
                reps = int(w) + (1 if rng.random() < w - int(w) else 0)
                self.items.extend([(di, s0, s1)] * max(1, reps))
        rng.shuffle(self.items)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        di, s0, s1 = self.items[i]
        return make_chunk(self.docs[di], s0, s1, self.context)

def collate_chunks(batch):
    wins = [w for it in batch for w in it["windows"]]
    W = len(wins)
    T = max(len(w["ids"]) for w in wins)
    M = max(len(w["mem"]) for w in wins)
    if M:
        M = XL_MEMORY + XL_MEMORY_OLD
    ids = torch.full((W, T), PAD_ID, dtype=torch.long); mask = torch.zeros((W, T), dtype=torch.bool)
    mem_ids = torch.full((W, M), PAD_ID, dtype=torch.long); mem_mask = torch.zeros((W, M), dtype=torch.bool)
    starts = torch.zeros(W, dtype=torch.float32)
    s_win, s_a, s_b, s_chunk, s_pos = [], [], [], [], []
    wi = 0
    for ci, it in enumerate(batch):
        for w in it["windows"]:
            n = len(w["ids"])
            ids[wi, :n] = torch.from_numpy(w["ids"]); mask[wi, :n] = True
            k = len(w["mem"])
            if k:
                mem_ids[wi, M - k:] = torch.from_numpy(w["mem"]); mem_mask[wi, M - k:] = True
            starts[wi] = w["start"]
            for (sp, a, b) in w["spans"]:
                s_win.append(wi); s_a.append(a); s_b.append(b); s_chunk.append(ci); s_pos.append(sp)
            wi += 1
    S = max(it["n"] for it in batch)
    labels = torch.full((len(batch), S), -100, dtype=torch.long)
    cmask = torch.zeros((len(batch), S), dtype=torch.bool)
    for ci, it in enumerate(batch):
        cmask[ci, :it["n"]] = True
        if "labels" in it:
            labels[ci, :it["n"]] = torch.from_numpy(it["labels"])
    t = lambda x: torch.tensor(x, dtype=torch.long)
    return dict(ids=ids, mask=mask, mem_ids=mem_ids, mem_mask=mem_mask, starts=starts,
                s_win=t(s_win), s_a=t(s_a), s_b=t(s_b), s_chunk=t(s_chunk), s_pos=t(s_pos),
                labels=labels, cmask=cmask, domain=t([it["domain"] for it in batch]))

def to_device(batch):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

_demo = make_chunk(DATA["train"][0], 0, min(CHUNK_SENTS, len(DATA["train"][0]["sentences"])), CONTEXT)
print(f"example chunk: {_demo['n']} sentences packed into {len(_demo['windows'])} window(s) of "
      f"{[len(w['ids']) for w in _demo['windows']]} tokens")

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, width, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(width))
        self.eps = eps

    def forward(self, x):
        norm = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * norm).to(x.dtype) * self.weight.to(x.dtype)

class RoPE(nn.Module):
    def __init__(self, dim, base=10_000.0):
        super().__init__()
        self.register_buffer("inv", 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim)), persistent=False)

    def rotate(self, x, pos):
        ang = pos.to(x.device, dtype=torch.float32).unsqueeze(-1) * self.inv
        cos = torch.cat((ang.cos(), ang.cos()), dim=-1).unsqueeze(1).to(x.dtype)
        sin = torch.cat((ang.sin(), ang.sin()), dim=-1).unsqueeze(1).to(x.dtype)
        half = x.size(-1) // 2
        return x * cos + torch.cat((-x[..., half:], x[..., :half]), -1) * sin

class GQA(nn.Module):
    def __init__(self, d, qh, kvh, dropout, rope_base):
        super().__init__()
        self.d, self.qh, self.kvh = d, qh, kvh
        self.dh = d // qh
        self.groups = qh // kvh
        self.dropout = dropout
        self.q = nn.Linear(d, qh * self.dh, bias=False)
        self.k = nn.Linear(d, kvh * self.dh, bias=False)
        self.v = nn.Linear(d, kvh * self.dh, bias=False)
        self.o = nn.Linear(d, d, bias=False)
        self.rope = RoPE(self.dh, rope_base)

    def project(self, x, pos):
        b, t, _ = x.shape
        q = self.q(x).view(b, t, self.qh, self.dh).transpose(1, 2)
        k = self.k(x).view(b, t, self.kvh, self.dh).transpose(1, 2)
        v = self.v(x).view(b, t, self.kvh, self.dh).transpose(1, 2)
        return self.rope.rotate(q, pos), self.rope.rotate(k, pos), v

    def project_memory(self, x, pos):
        b, t, _ = x.shape
        k = self.k(x).view(b, t, self.kvh, self.dh).transpose(1, 2)
        v = self.v(x).view(b, t, self.kvh, self.dh).transpose(1, 2)
        return self.rope.rotate(k, pos), v

    def attend(self, q, k, v, past, mem_mask, attn_mask):
        k = k.repeat_interleave(self.groups, dim=1)
        v = v.repeat_interleave(self.groups, dim=1)
        b, _, t, _ = q.shape
        cur = torch.ones(t, t, dtype=torch.bool, device=q.device).tril()[None, None].expand(b, 1, t, t)
        if attn_mask is not None:

            cur = (cur & attn_mask[:, None, None, :]) | torch.eye(t, dtype=torch.bool, device=q.device)[None, None]
        if past:
            pm = mem_mask[:, None, None, :].expand(b, 1, t, past)
            allowed = torch.cat([pm, cur], dim=-1)
        else:
            allowed = cur
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=self.dropout if self.training else 0.0)
        return self.o(out.transpose(1, 2).contiguous().view(b, t, self.d))

class Block(nn.Module):
    def __init__(self, d, qh, kvh, ff, dropout, rope_base, eps):
        super().__init__()
        self.n1 = RMSNorm(d, eps); self.attn = GQA(d, qh, kvh, dropout, rope_base); self.n2 = RMSNorm(d, eps)
        self.ff1 = nn.Linear(d, ff); self.ff2 = nn.Linear(ff, d)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, pos, mem=None, mem_pos=None, mem_mask=None, attn_mask=None):
        q, ck, cv = self.attn.project(self.n1(x), pos)
        past = 0
        if mem is not None and mem.size(1):
            pk, pv = self.attn.project_memory(self.n1(mem.to(x.dtype)), mem_pos)
            ck, cv, past = torch.cat((pk, ck), -2), torch.cat((pv, cv), -2), mem.size(1)
        x = x + self.drop(self.attn.attend(q, ck, cv, past, mem_mask, attn_mask))
        return x + self.drop(self.ff2(F.gelu(self.ff1(self.n2(x)))))

class PrefixEncoder(nn.Module):
    def __init__(self, n_layers, prefix_len, d, hidden):
        super().__init__()
        self.n_layers, self.prefix_len, self.d = n_layers, prefix_len, d
        self.seed = nn.Parameter(torch.randn(prefix_len, d) * 0.02)
        self.mlp = nn.Sequential(nn.Linear(d, hidden), nn.Tanh(), nn.Linear(hidden, n_layers * d))

    def forward(self):
        return self.mlp(self.seed).view(self.prefix_len, self.n_layers, self.d).transpose(0, 1)

class MultiResBackbone(nn.Module):
    def __init__(self, vocab, d, n_layers, qh, kvh, ff, dropout, rope_base=10_000.0, eps=1e-6):
        super().__init__()
        self.d_model = d
        self.emb = nn.Embedding(vocab, d)
        self.layers = nn.ModuleList([Block(d, qh, kvh, ff, dropout, rope_base, eps) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d, eps)
        self.prefix = None

    @property
    def blocks(self):
        return self.layers

    @property
    def token_embedding(self):
        return self.emb

    def add_prefix(self, prefix_len, hidden):
        self.prefix = PrefixEncoder(len(self.layers), prefix_len, self.d_model, hidden)

    def forward(self, input_ids, positions=None, mems=None, mem_pos=None, mem_mask=None, attn_mask=None,
                return_hidden=False, grad_ckpt=False):
        b, t = input_ids.shape
        if positions is None:
            positions = torch.arange(t, device=input_ids.device, dtype=torch.float32)[None].expand(b, -1)
        if attn_mask is None:
            attn_mask = torch.ones(b, t, dtype=torch.bool, device=input_ids.device)
        pre = self.prefix() if self.prefix is not None else None
        if pre is not None:
            P = pre.size(1)
            m_len = 0 if mems is None else mems[0].size(1)
            ppos = positions[:, :1] - m_len - P + torch.arange(P, device=input_ids.device, dtype=torch.float32)[None]
            pmask = torch.ones(b, P, dtype=torch.bool, device=input_ids.device)
            mem_pos = ppos if mems is None else torch.cat([ppos, mem_pos], 1)
            mem_mask = pmask if mems is None else torch.cat([pmask, mem_mask], 1)
        x = self.emb(input_ids)
        levels = [x]
        for i, layer in enumerate(self.layers):
            mem = None if mems is None else mems[i]
            if pre is not None:
                p_i = pre[i][None].expand(b, -1, -1).to(x.dtype)
                mem = p_i if mem is None else torch.cat([p_i, mem.to(x.dtype)], 1)
            if grad_ckpt:
                x = checkpoint(layer, x, positions, mem, mem_pos, mem_mask, attn_mask, use_reentrant=False)
            else:
                x = layer(x, positions, mem, mem_pos, mem_mask, attn_mask)
            levels.append(x)
        out = self.final_norm(x)
        return (out, levels) if return_hidden else out

def _layer_params(c):
    d, qh, kvh, ff = c["d_model"], c["query_heads"], c["kv_heads"], c["ff_dim"]
    dh = d // qh
    return d * d * 2 + 2 * d * kvh * dh + d * ff + ff + ff * d + d + 2 * d

def backbone_params(n_layers):
    return P1_CFG["vocab_size"] * P1_CFG["d_model"] + n_layers * _layer_params(P1_CFG) + P1_CFG["d_model"]

P1_LAYERS_USED = P1_CFG["layers"]
if backbone_params(P1_LAYERS_USED) > MAX_PARAMS and not SMOKE_TEST:
    if not AUTO_TRIM_LAYERS:
        raise RuntimeError(f"Full Phase-1 model has {backbone_params(P1_LAYERS_USED)/1e6:.1f}M parameters > limit.")
    while P1_LAYERS_USED > 1 and backbone_params(P1_LAYERS_USED) > MAX_PARAMS:
        P1_LAYERS_USED -= 1
    print(f"NOTE: the full Phase-1 model ({P1_CFG['layers']} layers) has {backbone_params(P1_CFG['layers'])/1e6:.2f}M "
          f"parameters > {MAX_PARAMS/1e6:.0f}M limit -> using the first {P1_LAYERS_USED} layers "
          f"({backbone_params(P1_LAYERS_USED)/1e6:.2f}M). The dropped top layer(s) are the most LM-specific.")

def build_backbone(n_layers=None, layer_map=None, dropout=None):
    c = P1_CFG
    lmap = layer_map if layer_map is not None else list(range(n_layers or P1_LAYERS_USED))
    bb = MultiResBackbone(VOCAB_SIZE, c["d_model"], len(lmap), c["query_heads"], c["kv_heads"], c["ff_dim"],
                          BACKBONE_DROPOUT if dropout is None else dropout, c.get("rope_base", 10_000.0),
                          c.get("rms_eps", 1e-6))
    sd = {k: v for k, v in P1_STATE.items() if not k.startswith("lm_head")}
    new_sd = {k: v for k, v in sd.items() if not k.startswith("layers.")}
    for new_i, old_i in enumerate(lmap):
        pre = f"layers.{old_i}."
        for k, v in sd.items():
            if k.startswith(pre):
                new_sd[f"layers.{new_i}." + k[len(pre):]] = v
    missing, unexpected = bb.load_state_dict(new_sd, strict=False)
    if missing or unexpected:
        raise RuntimeError(f"Phase-1 checkpoint does not match backbone. missing={missing[:5]} unexpected={unexpected[:5]}")
    return bb

def build_student_backbone(cfg, drop):
    n = cfg.get("student_layers") or P1_LAYERS_USED
    lm = [int(round(x)) for x in np.linspace(0, P1_LAYERS_USED - 1, n)]
    return build_backbone(layer_map=lm, dropout=drop), lm

class TokenEncoder(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone, self.out_dim = backbone, backbone.d_model

    def _memory(self, mem_ids, mem_mask, mem_pos):
        with torch.no_grad():
            _, lv = self.backbone(mem_ids, positions=mem_pos, attn_mask=mem_mask, return_hidden=True)
        b, M = mem_ids.shape
        O = M - min(M, XL_MEMORY)
        G = P1_CFG["compress_group_size"]
        w = mem_mask[:, :O].float().view(b, O // G, G) if O else None
        mems = []
        for x in lv[:-1]:
            x = x.detach()
            if O:
                pooled = (x[:, :O].view(b, O // G, G, -1) * w[..., None]).sum(2) / w.sum(2).clamp(min=1)[..., None]
                x = torch.cat([pooled.to(x.dtype), x[:, O:]], 1)
            mems.append(x)
        if O:
            ppos = (mem_pos[:, :O].view(b, O // G, G) * w).sum(2) / w.sum(2).clamp(min=1)
            return mems, torch.cat([ppos, mem_pos[:, O:]], 1), torch.cat([w.sum(2) > 0, mem_mask[:, O:]], 1)
        return mems, mem_pos, mem_mask

    def forward(self, ids, mask, mem_ids=None, mem_mask=None, starts=None, return_hidden=False, grad_ckpt=False):
        b, t = ids.shape
        if starts is None:
            starts = torch.zeros(b, device=ids.device)
        positions = starts.float()[:, None] + torch.arange(t, device=ids.device, dtype=torch.float32)[None]
        mems = mem_pos = mm = None
        if mem_ids is not None and mem_ids.size(1) > 0:
            M = mem_ids.size(1)
            mp = starts.float()[:, None] - M + torch.arange(M, device=ids.device, dtype=torch.float32)[None]
            mems, mem_pos, mm = self._memory(mem_ids, mem_mask, mp)
        out = self.backbone(ids, positions=positions, mems=mems, mem_pos=mem_pos, mem_mask=mm, attn_mask=mask,
                            return_hidden=return_hidden, grad_ckpt=grad_ckpt)
        if return_hidden:
            return out[0], out[1][1:]
        return out

_probe = build_backbone()
P1_PARAMS = sum(p.numel() for p in _probe.parameters())
print(f"Phase-1 multi-res decoder used: {P1_LAYERS_USED} layers, {P1_PARAMS/1e6:.2f}M parameters")
assert MIN_PARAMS <= P1_PARAMS <= MAX_PARAMS or SMOKE_TEST
del _probe

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout=0.0, rank_stabilised=False):
        super().__init__()
        self.base = base
        self.scale = alpha / (math.sqrt(r) if rank_stabilised else r)
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)

class LoRMAPlusLinear(nn.Module):
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)
        y = y0 + self.scale * F.linear(F.linear(y0, self.A), self.B)
        return y if self.base.bias is None else y + self.base.bias

class LoRMAPiLinear(nn.Module):
    def __init__(self, base, r, alpha=None, dropout=0.0):
        super().__init__()
        self.base = base
        out = base.out_features
        B = torch.randn(out, r) / math.sqrt(r); B[:, 0] = 1.0
        A = torch.zeros(r, out); A[0, 0] = 1.0
        self.B, self.A = nn.Parameter(B), nn.Parameter(A)
        self.register_buffer("rot_idx", (torch.arange(out)[None, :] - torch.arange(out)[:, None]) % out, persistent=False)

    def forward(self, x):
        m_pi = torch.gather(torch.matmul(self.B, self.A), 1, self.rot_idx)
        y = F.linear(F.linear(x, self.base.weight), m_pi)
        return y if self.base.bias is None else y + self.base.bias

class IA3Out(nn.Module):
    def __init__(self, base, *args, **kw):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.l.to(x.dtype)

class IA3In(nn.Module):
    def __init__(self, base, *args, **kw):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l.to(x.dtype))

NF4_LEVELS = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
                           -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
                           0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
                           0.7229568362236023, 1.0])
QBLOCK = 64

def nf4_quantize(w):
    flat = w.detach().float().reshape(-1)
    pad = (-flat.numel()) % QBLOCK
    flat = torch.cat([flat, flat.new_zeros(pad)])
    blocks = flat.view(-1, QBLOCK)
    absmax = blocks.abs().max(1).values.clamp(min=1e-8)
    codes = (blocks / absmax[:, None]).unsqueeze(-1).sub(NF4_LEVELS).abs().argmin(-1).to(torch.uint8).view(-1)
    packed = (codes[0::2] | (codes[1::2] << 4)).contiguous()
    return packed, absmax.half(), pad

class QLoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.out_features, self.in_features = base.out_features, base.in_features
        packed, absmax, pad = nf4_quantize(base.weight)
        self.register_buffer("qweight", packed)
        self.register_buffer("absmax", absmax)
        self.register_buffer("levels", NF4_LEVELS.clone())
        self.pad = pad
        self.bias = None if base.bias is None else nn.Parameter(base.bias.detach().clone(), requires_grad=False)
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, self.in_features))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def dequantize(self, dtype):
        codes = torch.stack([self.qweight & 15, self.qweight >> 4], 1).view(-1).long()
        w = (self.levels[codes].view(-1, QBLOCK) * self.absmax.float()[:, None]).view(-1)
        if self.pad:
            w = w[:-self.pad]
        return w.view(self.out_features, self.in_features).to(dtype)

    def forward(self, x):

        y = F.linear(x, self.dequantize(torch.float32), self.bias)
        return y + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)

LOWRANK_CLASSES = {"lora": LoRALinear, "rslora": LoRALinear, "lorma_plus": LoRMAPlusLinear,
                   "lorma_pi": LoRMAPiLinear, "qlora": QLoRALinear}
ADAPTER_TYPES = (LoRALinear, LoRMAPlusLinear, LoRMAPiLinear, QLoRALinear, IA3Out, IA3In)

def _resolve(block, path):
    parent = block
    parts = path.split(".")
    for p in parts[:-1]:
        parent = getattr(parent, p)
    return parent, parts[-1], getattr(parent, parts[-1])

def inject(backbone, cfg):
    method, n = cfg["method"], 0
    for block in backbone.blocks:
        if method == "ia3":
            for path, where in IA3_TARGETS.items():
                try:
                    parent, name, leaf = _resolve(block, path)
                except AttributeError:
                    continue
                if isinstance(leaf, nn.Linear):
                    setattr(parent, name, (IA3Out if where == "out" else IA3In)(leaf)); n += 1
            continue
        cls = LOWRANK_CLASSES[method]
        for path in LOWRANK_TARGETS:
            try:
                parent, name, leaf = _resolve(block, path)
            except AttributeError:
                continue
            if not isinstance(leaf, nn.Linear):
                continue
            if method in ("lora", "rslora"):
                new = LoRALinear(leaf, cfg["rank"], cfg["alpha"], cfg.get("lora_dropout", 0.0), method == "rslora")
            else:
                new = cls(leaf, cfg["rank"], cfg.get("alpha", cfg["rank"]), cfg.get("lora_dropout", 0.0))
            setattr(parent, name, new); n += 1
    print(f"{method}: adapted {n} linear layers")
    return n

def configure_trainable(model, cfg):
    method = cfg["method"]
    for p in model.parameters():
        p.requires_grad = False
    bb = model.encoder.backbone
    if method in ("full", "distill"):
        for p in model.parameters():
            p.requires_grad = True
    elif method in LOWRANK_CLASSES:
        for mod in bb.modules():
            if isinstance(mod, (LoRALinear, LoRMAPlusLinear, LoRMAPiLinear, QLoRALinear)):
                mod.A.requires_grad = True; mod.B.requires_grad = True
    elif method == "ia3":
        for mod in bb.modules():
            if isinstance(mod, (IA3Out, IA3In)):
                mod.l.requires_grad = True
    elif method == "prefix":
        for p in bb.prefix.parameters():
            p.requires_grad = True
    else:
        raise ValueError(method)
    for m in model.head_modules():
        for p in m.parameters():
            p.requires_grad = True

def build_rr_model(cfg):
    method = cfg["method"]
    drop = cfg["dropout"] if cfg.get("dropout") is not None else BACKBONE_DROPOUT
    if method == "distill":
        backbone, lm = build_student_backbone(cfg, drop)
        cfg["layer_map"] = lm
    else:
        backbone = build_backbone(dropout=drop)
    if method in LOWRANK_CLASSES or method == "ia3":
        inject(backbone, cfg)
    elif method == "prefix":
        if not hasattr(backbone, "add_prefix"):
            raise ValueError("prefix tuning is implemented for the multi-resolution memory model only")
        backbone.add_prefix(cfg["prefix_len"], cfg["prefix_hidden"])
    cfg["context"] = cfg.get("context") or CONTEXT
    cfg["head"] = cfg.get("head") or ("bilstm_crf" if cfg["context"] == "document" else "linear")
    model = RRModel(TokenEncoder(backbone), cfg["context"], cfg["head"])
    model.grad_ckpt = USE_GRAD_CHECKPOINTING
    configure_trainable(model, cfg)
    return model

def _adapter_checks():
    torch.manual_seed(0)
    base = nn.Linear(24, 128)
    x = torch.randn(3, 24)
    ref = base(x)
    for name, m in [("lora", LoRALinear(base, 4, 8)), ("rslora", LoRALinear(base, 4, 8, rank_stabilised=True)),
                    ("lorma_plus", LoRMAPlusLinear(base, 4, 4)), ("lorma_pi", LoRMAPiLinear(base, 4)),
                    ("ia3_out", IA3Out(base))]:
        assert torch.allclose(m(x), ref, atol=1e-5), f"{name} not identity at init"
    base2 = nn.Linear(128, 24)
    assert torch.allclose(IA3In(base2)(ref), base2(ref), atol=1e-5)
    q = QLoRALinear(base, 4, 8)
    rel = (q.dequantize(torch.float32) - base.weight).norm() / base.weight.norm()
    assert rel < 0.15, f"NF4 reconstruction error too high: {rel:.3f}"
    print(f"Adapter checks passed (identity at init; NF4 4-bit relative weight error {rel:.3f}).")

_adapter_checks()

In [ ]:
class CRF(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.trans = nn.Parameter(torch.zeros(n, n))
        self.start = nn.Parameter(torch.zeros(n))
        self.end = nn.Parameter(torch.zeros(n))

    def nll(self, em, tags, mask):
        tags = tags.clamp(min=0)
        mf = mask.float()
        B, S, K = em.shape
        score = self.start[tags[:, 0]] + em[:, 0].gather(1, tags[:, :1]).squeeze(1)
        if S > 1:
            score = score + ((em[:, 1:].gather(2, tags[:, 1:, None]).squeeze(2) +
                              self.trans[tags[:, :-1], tags[:, 1:]]) * mf[:, 1:]).sum(1)
        last = tags.gather(1, (mask.sum(1) - 1).clamp(min=0)[:, None]).squeeze(1)
        score = score + self.end[last]
        alpha = self.start[None] + em[:, 0]
        for t in range(1, S):
            nxt = torch.logsumexp(alpha[:, :, None] + self.trans[None] + em[:, t, None, :], dim=1)
            alpha = torch.where(mask[:, t, None], nxt, alpha)
        return torch.logsumexp(alpha + self.end[None], dim=1) - score

class RRModel(nn.Module):
    def __init__(self, encoder, context, head):
        super().__init__()
        self.encoder, self.context, self.head_type = encoder, context, head
        pooled = encoder.out_dim * (2 if POOLING == "mean+last" else 1)
        self.domain_emb = nn.Embedding(len(DOMAINS), DOMAIN_EMB_DIM)
        self.proj = nn.Sequential(nn.Dropout(HEAD_DROPOUT), nn.Linear(pooled + DOMAIN_EMB_DIM, SENT_DIM), nn.GELU())
        if head in ("bilstm", "bilstm_crf"):
            self.lstm = nn.LSTM(SENT_DIM, LSTM_HIDDEN, num_layers=1, batch_first=True, bidirectional=True)
            out = 2 * LSTM_HIDDEN + SENT_DIM
        else:
            self.lstm, out = None, SENT_DIM
        self.drop = nn.Dropout(HEAD_DROPOUT)
        self.classifier = nn.Linear(out, NUM_LABELS)
        self.crf = CRF(NUM_LABELS) if head == "bilstm_crf" else None
        self.grad_ckpt = False

    def head_modules(self):
        return [m for m in (self.domain_emb, self.proj, self.lstm, self.classifier, self.crf) if m is not None]

    def forward(self, b, return_hidden=False):
        out = self.encoder(b["ids"], b["mask"], b["mem_ids"], b["mem_mask"], b["starts"],
                           return_hidden=return_hidden, grad_ckpt=self.grad_ckpt and self.training)
        h, levels = out if return_hidden else (out, None)
        hf = h.float()
        w, a, e = b["s_win"], b["s_a"], b["s_b"]
        cs = torch.cat([hf.new_zeros(hf.size(0), 1, hf.size(-1)), hf.cumsum(1)], 1)
        sv = (cs[w, e] - cs[w, a]) / (e - a).clamp(min=1).float()[:, None]
        if POOLING == "mean+last":
            sv = torch.cat([sv, hf[w, e - 1]], -1)
        Bc, S = b["cmask"].shape
        grid = sv.new_zeros(Bc, S, sv.size(-1))
        grid[b["s_chunk"], b["s_pos"]] = sv
        dom = self.domain_emb(b["domain"])[:, None].expand(-1, S, -1)
        x = self.proj(torch.cat([grid, dom.float()], -1))
        if self.lstm is not None:
            lengths = b["cmask"].sum(1).clamp(min=1).cpu()
            packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
            y, _ = self.lstm(packed)
            y, _ = nn.utils.rnn.pad_packed_sequence(y, batch_first=True, total_length=S)
            x = torch.cat([self.drop(y), x.to(y.dtype)], -1)
        em = self.classifier(x)
        return (em, levels) if return_hidden else em

@torch.no_grad()
def predict_doc_scores(model, doc):
    model.eval()
    n = len(doc["sentences"])
    acc = np.zeros((n, NUM_LABELS), np.float32)
    wsum = np.zeros(n, np.float32)
    starts = chunk_starts_eval(n)
    for k in range(0, len(starts), EVAL_BATCH_CHUNKS):
        items = [make_chunk(doc, s, min(n, s + CHUNK_SENTS), model.context, with_labels=False)
                 for s in starts[k:k + EVAL_BATCH_CHUNKS]]
        b = to_device(collate_chunks(items))
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
            em = model(b)
        em = em.float().cpu().numpy()
        for j, it in enumerate(items):
            m = it["n"]
            pos = np.arange(m)
            wt = np.minimum(np.minimum(pos + 1, m - pos), max(1, CHUNK_STRIDE)).astype(np.float32)
            acc[it["s0"]:it["s0"] + m] += em[j, :m] * wt[:, None]
            wsum[it["s0"]:it["s0"] + m] += wt
    return acc / np.maximum(wsum, 1e-6)[:, None]

def model_transitions(model):
    if getattr(model, "crf", None) is None:
        z = np.zeros(NUM_LABELS, np.float32)
        return np.zeros((NUM_LABELS, NUM_LABELS), np.float32), z, z
    return tuple(p.detach().float().cpu().numpy() for p in (model.crf.trans, model.crf.start, model.crf.end))

def viterbi(sc, pack):
    trans, start, end = pack
    n = sc.shape[0]
    if n == 0:
        return np.zeros(0, np.int64)
    score = sc[0] + start
    back = np.zeros((n, NUM_LABELS), np.int64)
    for t in range(1, n):
        cand = score[:, None] + trans
        back[t] = cand.argmax(0)
        score = cand.max(0) + sc[t]
    score = score + end
    out = np.empty(n, np.int64)
    out[-1] = score.argmax()
    for t in range(n - 1, 0, -1):
        out[t - 1] = back[t, out[t]]
    return out

def decode(sc, pack, method):
    return viterbi(sc, pack) if method == "viterbi" else sc.argmax(-1)

def macro_scores(gold, pred):
    gold, pred = np.asarray(gold), np.asarray(pred)
    used = sorted(set(gold.tolist()) | set(pred.tolist()))
    per = {}
    for l in range(NUM_LABELS):
        tp = int(((pred == l) & (gold == l)).sum()); fp = int(((pred == l) & (gold != l)).sum())
        fn = int(((pred != l) & (gold == l)).sum())
        P = tp / (tp + fp) if tp + fp else 0.0
        R = tp / (tp + fn) if tp + fn else 0.0
        per[LABELS[l]] = dict(precision=P, recall=R, f1=2 * P * R / (P + R) if P + R else 0.0, support=tp + fn)
    m = lambda k: float(np.mean([per[LABELS[l]][k] for l in used])) if used else 0.0
    return dict(macro_f1=m("f1"), macro_precision=m("precision"), macro_recall=m("recall"),
                accuracy=float((gold == pred).mean()) if len(gold) else 0.0, per_label=per)

def evaluate_docs(model, docs, methods=("viterbi",), keep_scores=False):
    pack = model_transitions(model)
    preds = {m: {} for m in methods}
    scores = {}
    t0, n_sent = time.time(), 0
    for d in docs:
        sc = predict_doc_scores(model, d)
        n_sent += len(d["sentences"])
        for m in methods:
            preds[m][d["id"]] = decode(sc, pack, m)
        if keep_scores:
            scores[d["id"]] = sc
    dt = time.time() - t0
    res = {}
    for m in methods:
        res[m] = {}
        for dom in ["all"] + DOMAINS:
            sel = [d for d in docs if dom == "all" or d["domain"] == dom]
            if not sel:
                continue
            g = np.concatenate([d["labels"] for d in sel]); p = np.concatenate([preds[m][d["id"]] for d in sel])
            res[m][dom] = macro_scores(g, p)
    return res, preds, dict(sentences=n_sent, seconds=dt, sentences_per_s=n_sent / max(dt, 1e-9)), scores

In [ ]:
TRAINED_MODELS, RESULTS = {}, {}

def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if (p.requires_grad or not trainable_only))

def logical_params(module):
    q = sum(m.out_features * m.in_features for m in module.modules() if isinstance(m, QLoRALinear))
    return count_params(module) + q

def make_cfg(name):
    cfg = dict(EXPERIMENT_DEFAULTS)
    cfg.update(copy.deepcopy(EXPERIMENTS[name]))
    cfg["name"] = name
    return cfg

def class_weights():
    if CLASS_WEIGHT == "none":
        return None
    c = TRAIN_LABEL_COUNTS + 1.0
    return torch.tensor(np.clip(np.sqrt(c.max() / c), 1.0, 10.0), dtype=torch.float32, device=DEVICE)

def trainable_state(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in names}

def load_trained(name):
    run_dir = OUTPUT_DIR / "runs" / name
    cfg = json.loads((run_dir / "cfg.json").read_text())
    model = build_rr_model(cfg)
    state = torch.load(run_dir / "best_state.pt", map_location="cpu")
    model.load_state_dict(state, strict=False)
    assert {n for n, p in model.named_parameters() if p.requires_grad} <= set(state)
    return model

def rr_loss(model, em, b, cw):
    mask, labels = b["cmask"], b["labels"]
    ce = F.cross_entropy(em[mask].float(), labels[mask], weight=cw)
    if model.crf is None:
        return ce
    return model.crf.nll(em.float(), labels, mask).sum() / mask.sum().clamp(min=1) + CRF_AUX_CE_WEIGHT * ce

def kd_loss(model, em, t_ems, s_hid, t_hid, b, cfg, cw):
    mask = b["cmask"]
    T = cfg["kd_temperature"]
    target = torch.stack([F.softmax(t[mask].float() / T, -1) for t in t_ems]).mean(0)
    kd = F.kl_div(F.log_softmax(em[mask].float() / T, -1), target, reduction="batchmean") * T * T
    hard = rr_loss(model, em, b, cw)
    hid = em.new_zeros(()).float()
    if cfg["kd_hidden_weight"] > 0 and s_hid:
        tok = b["mask"]
        for si, ti in enumerate(cfg["layer_map"]):
            x = F.layer_norm(s_hid[si].float(), (s_hid[si].size(-1),))
            y = F.layer_norm(t_hid[ti].float(), (t_hid[ti].size(-1),))
            hid = hid + ((x - y) ** 2).mean(-1)[tok].mean()
        hid = hid / len(cfg["layer_map"])
    return (1 - cfg["kd_alpha"]) * kd + cfg["kd_alpha"] * hard + cfg["kd_hidden_weight"] * hid

def labels_of(pred_ids):
    return [LABELS[int(i)] for i in pred_ids]

def write_submission(run_dir, preds_by_doc):
    sub = {}
    for d in DATA["test"]:
        p = labels_of(preds_by_doc[d["id"]])
        assert len(p) == len(d["sentences"]), f"{d['id']}: {len(p)} predictions for {len(d['sentences'])} sentences"
        assert all(l in LABEL2ID for l in p)
        sub[d["id"]] = p
    assert set(sub) == set(test_ids)
    pj = Path(run_dir) / "prediction.json"
    pj.write_text(json.dumps(sub, ensure_ascii=False), encoding="utf-8")
    zp = Path(run_dir) / "submission.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    return zp

def save_scores(path, scores, pack):
    arrs = {f"doc_{k}": v.astype(np.float16) for k, v in scores.items()}
    arrs["__trans"], arrs["__start"], arrs["__end"] = pack
    np.savez_compressed(path, **arrs)

def run_experiment(name, teacher=None):
    cfg = make_cfg(name)
    run_dir = OUTPUT_DIR / "runs" / name
    run_dir.mkdir(parents=True, exist_ok=True)
    if RESUME_COMPLETED_RUNS and (run_dir / "results.json").exists():
        RESULTS[name] = json.loads((run_dir / "results.json").read_text())
        print(f"[{name}] already finished (dev macro-F1 {RESULTS[name]['dev']['all']['macro_f1']:.4f}) - skipping")
        return RESULTS[name]
    print("=" * 100); print(f"{name}: {cfg['description']}"); print("=" * 100)
    seed_everything(SEED)
    is_kd = cfg["method"] == "distill"
    model = build_rr_model(cfg).to(DEVICE)
    n_bb = logical_params(model.encoder)
    n_total, n_train = logical_params(model), count_params(model, True)
    print(f"backbone (+adapters) {n_bb/1e6:.2f}M | whole model {n_total/1e6:.2f}M | trainable {n_train/1e6:.3f}M "
          f"({100*n_train/n_total:.2f}%) | context={cfg['context']} head={cfg['head']}")
    assert MIN_PARAMS <= n_bb <= MAX_PARAMS or SMOKE_TEST, \
        f"backbone {n_bb/1e6:.1f}M outside the {MIN_PARAMS/1e6:.0f}M-{MAX_PARAMS/1e6:.0f}M limit"
    teachers = [] if teacher is None else (list(teacher) if isinstance(teacher, (list, tuple)) else [teacher])
    if is_kd:
        assert teachers, "distillation needs a teacher"
        for t in teachers:
            t.to(DEVICE).eval()
            for p in t.parameters():
                p.requires_grad = False
        print(f"teacher(s): {len(teachers)} model(s), {sum(logical_params(t.encoder) for t in teachers)/1e6:.1f}M "
              f"backbone parameters -> student {n_bb/1e6:.1f}M")
    head_ids = {id(p) for m in model.head_modules() for p in m.parameters()}
    head = [p for p in model.parameters() if p.requires_grad and id(p) in head_ids]
    body = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    decay = cfg["method"] in ("full", "distill")
    groups = [dict(params=head, lr=cfg["head_lr"], weight_decay=0.0)]
    if body:
        groups.append(dict(params=body, lr=cfg["lr"], weight_decay=WEIGHT_DECAY if decay else 0.0))
    optim = torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-8)
    bs, accum = cfg["batch_chunks"] or BATCH_CHUNKS, cfg["grad_accum"] or GRAD_ACCUM
    rng = random.Random(SEED)
    ds = TrainChunks(DATA["train"], cfg["context"], rng)
    total = max(1, math.ceil(len(ds) / (bs * accum)) * cfg["epochs"])
    warm = max(1, int(WARMUP_FRACTION * total))
    sched = torch.optim.lr_scheduler.LambdaLR(optim, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    use_amp = cfg["use_amp"] and DEVICE.type == "cuda"
    scaler = make_grad_scaler(use_amp)
    cw = class_weights()
    params = [p for g in groups for p in g["params"]]
    print(f"train chunks/epoch ~{len(ds):,} | optimizer steps ~{total}")
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    history, best, best_ep, bad, t0 = [], -1.0, 0, 0, time.time()

    def step():
        if scaler is not None:
            scaler.unscale_(optim)
        torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
        if scaler is None:
            optim.step()
        else:
            scaler.step(optim); scaler.update()
        sched.step(); optim.zero_grad(set_to_none=True)

    for ep in range(1, cfg["epochs"] + 1):
        if ep > 1:
            ds = TrainChunks(DATA["train"], cfg["context"], rng)
        loader = DataLoader(ds, batch_size=bs, shuffle=False, collate_fn=collate_chunks, num_workers=0)
        model.train()
        run_loss, n_loss, micro = 0.0, 0, 0
        optim.zero_grad(set_to_none=True)
        for b in tqdm(loader, desc=f"{name} ep{ep}", leave=False):
            b = to_device(b)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                if is_kd:
                    em, s_hid = model(b, return_hidden=True)
                    with torch.no_grad():
                        t_out = [t(b, return_hidden=True) for t in teachers]
                    loss = kd_loss(model, em, [o[0] for o in t_out], s_hid, t_out[0][1], b, cfg, cw)
                else:
                    loss = rr_loss(model, model(b), b, cw)
            if not torch.isfinite(loss):
                optim.zero_grad(set_to_none=True); continue
            (loss / accum).backward() if scaler is None else scaler.scale(loss / accum).backward()
            micro += 1; run_loss += float(loss); n_loss += 1
            if micro % accum == 0:
                step()
        if micro % accum:
            step()
        res, _, _, _ = evaluate_docs(model, DATA["dev"])
        f1 = res["viterbi"]["all"]["macro_f1"]
        history.append(dict(epoch=ep, train_loss=run_loss / max(1, n_loss), dev_macro_f1=f1,
                            dev_cl=res["viterbi"].get("CL", {}).get("macro_f1"),
                            dev_it=res["viterbi"].get("IT", {}).get("macro_f1"), elapsed_s=time.time() - t0))
        print(f"  epoch {ep:2d} | loss {history[-1]['train_loss']:.4f} | dev macro-F1 {f1:.4f} "
              f"(CL {history[-1]['dev_cl'] or 0:.4f} / IT {history[-1]['dev_it'] or 0:.4f}) | {(time.time()-t0)/60:.1f} min")
        if f1 > best + 1e-5:
            best, best_ep, bad = f1, ep, 0
            torch.save(trainable_state(model), run_dir / "best_state.pt")
        elif best > 0:
            bad += 1
            if bad >= cfg["patience"]:
                print(f"  early stop (no improvement for {cfg['patience']} epochs)")
                break
    train_time = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 1024**3 if DEVICE.type == "cuda" else 0.0
    model.load_state_dict(torch.load(run_dir / "best_state.pt", map_location=DEVICE), strict=False)
    res, preds, speed, dev_sc = evaluate_docs(model, DATA["dev"], methods=("viterbi", "greedy"), keep_scores=True)
    decoder = max(res, key=lambda m: res[m]["all"]["macro_f1"])
    dev = res[decoder]
    print(f"[{name}] best epoch {best_ep} | decoder={decoder} | dev macro-P/R/F1 = {dev['all']['macro_precision']:.4f}/"
          f"{dev['all']['macro_recall']:.4f}/{dev['all']['macro_f1']:.4f} | CL {dev.get('CL', {}).get('macro_f1', 0):.4f} "
          f"| IT {dev.get('IT', {}).get('macro_f1', 0):.4f} | {speed['sentences_per_s']:.0f} sent/s")
    pack = model_transitions(model)
    save_scores(run_dir / "dev_scores.npz", dev_sc, pack)
    save_json(run_dir / "dev_predictions.json", {k: labels_of(v) for k, v in preds[decoder].items()})
    out = dict(name=name, method=cfg["method"], description=cfg["description"], cfg=cfg, context=cfg["context"],
               params_backbone=n_bb, params_total=n_total, params_trainable=n_train,
               trainable_pct=100 * n_train / n_total, train_time_s=train_time, best_epoch=best_ep,
               epochs_run=len(history), peak_mem_gb=peak, decoder=decoder, dev=dev,
               dev_greedy=res["greedy"], dev_viterbi=res["viterbi"], sentences_per_s=speed["sentences_per_s"],
               state_mb=(run_dir / "best_state.pt").stat().st_size / 1024**2, history=history)
    if WRITE_SUBMISSIONS:
        test_pred, test_sc = {}, {}
        for d in DATA["test"]:
            sc = predict_doc_scores(model, d)
            test_sc[d["id"]] = sc
            test_pred[d["id"]] = decode(sc, pack, decoder)
        save_scores(run_dir / "test_scores.npz", test_sc, pack)
        out["submission_zip"] = str(write_submission(run_dir, test_pred))
        print(f"        submission: {out['submission_zip']}")
    save_json(run_dir / "cfg.json", cfg)
    save_json(run_dir / "results.json", out)
    RESULTS[name] = out
    model.cpu(); TRAINED_MODELS[name] = model
    for t in teachers:
        t.cpu()
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return out

In [ ]:
for _p in sorted((OUTPUT_DIR / "runs").glob("*/results.json")):
    RESULTS[_p.parent.name] = json.loads(_p.read_text())

TEACHER_CANDIDATES = [n for n, c in EXPERIMENTS.items() if c["method"] != "distill"]

def pick_teachers(k):
    done = {n: RESULTS[n]["dev"]["all"]["macro_f1"] for n in TEACHER_CANDIDATES if n in RESULTS}
    if not done:
        raise RuntimeError("Distillation needs a finished teacher: run the other experiments first.")
    names = sorted(done, key=done.get, reverse=True)[:k]
    return names, [TRAINED_MODELS.get(n) or load_trained(n) for n in names]

for name in RUN_LIST:
    teacher = None
    if EXPERIMENTS[name]["method"] == "distill":
        tnames, teacher = pick_teachers(EXPERIMENTS[name].get("teacher_ensemble", 1))
        EXPERIMENTS[name]["teacher_name"] = "+".join(tnames)
        print(f"Teacher(s) for {name}: " + ", ".join(f"{n} (dev macro-F1 {RESULTS[n]['dev']['all']['macro_f1']:.4f})"
                                                     for n in tnames))
    run_experiment(name, teacher)
    gc.collect()
print("\nFinished:", [n for n in RUN_LIST if n in RESULTS])

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
ORDER = [n for n in EXPERIMENTS if n in RESULTS]
rows = []
for n in ORDER:
    r = RESULTS[n]; d = r["dev"]
    rows.append(dict(experiment=n, method=r["method"], context=r["context"], backbone_M=r["params_backbone"] / 1e6,
                     trainable_M=r["params_trainable"] / 1e6, trainable_pct=r["trainable_pct"],
                     dev_macro_F1=d["all"]["macro_f1"], dev_macro_P=d["all"]["macro_precision"],
                     dev_macro_R=d["all"]["macro_recall"], CL_F1=d.get("CL", {}).get("macro_f1"),
                     IT_F1=d.get("IT", {}).get("macro_f1"), decoder=r["decoder"],
                     viterbi_F1=r["dev_viterbi"]["all"]["macro_f1"], greedy_F1=r["dev_greedy"]["all"]["macro_f1"],
                     train_min=r["train_time_s"] / 60, peak_mem_GB=r["peak_mem_gb"], sent_per_s=r["sentences_per_s"],
                     ckpt_MB=r["state_mb"]))
summary = pd.DataFrame(rows).set_index("experiment")
summary.to_csv(OUTPUT_DIR / f"phase3_{ARCH_NAME}_results.csv")
display(summary.round(4))

per_label = pd.DataFrame({n: {l: RESULTS[n]["dev"]["all"]["per_label"][l]["f1"] for l in LABELS} for n in ORDER})
per_label["dev_support"] = [RESULTS[ORDER[0]]["dev"]["all"]["per_label"][l]["support"] for l in LABELS]
display(per_label.round(3))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
summary[["CL_F1", "IT_F1", "dev_macro_F1"]].plot.bar(ax=ax[0]); ax[0].set_title("Dev macro-F1 (CL / IT / all)")
ax[0].tick_params(axis="x", rotation=30)
for n in ORDER:
    h = RESULTS[n]["history"]
    ax[1].plot([x["epoch"] for x in h], [x["dev_macro_f1"] for x in h], marker="o", label=n)
ax[1].set_title("Dev macro-F1 per epoch"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "phase3_overview.png", dpi=130); plt.show()

best_name = max(ORDER, key=lambda n: RESULTS[n]["dev"]["all"]["macro_f1"])
dev_pred = json.loads((OUTPUT_DIR / "runs" / best_name / "dev_predictions.json").read_text())
cm = np.zeros((NUM_LABELS, NUM_LABELS), np.int64)
for d in DATA["dev"]:
    for g, p in zip(d["labels"], dev_pred[d["id"]]):
        cm[g, LABEL2ID[p]] += 1
cm_df = pd.DataFrame(cm, index=[f"gold:{l}" for l in LABELS], columns=LABELS)
print(f"Confusion matrix on dev for the best run ({best_name}):")
display(cm_df)
pairs = [(cm[i, j], LABELS[i], LABELS[j]) for i in range(NUM_LABELS) for j in range(NUM_LABELS) if i != j and cm[i, j]]
print("Most frequent confusions (gold -> predicted):")
for c, g, p in sorted(pairs, reverse=True)[:10]:
    print(f"   {g:24s} -> {p:24s} {c}")

print("\n--- Key comparisons for the written analysis ---")
ref_name = best_name
ref = RESULTS[ref_name]["dev"]["all"]["macro_f1"]
for n in ORDER:
    r = RESULTS[n]
    if n != ref_name and ref > 0:
        print(f"{n:28s}: {r['dev']['all']['macro_f1']:.4f} = {r['dev']['all']['macro_f1']/ref:.1%} of {ref_name} "
              f"with {r['trainable_pct']:.2f}% trainable")
doc_runs = [n for n in ORDER if RESULTS[n]["context"] == "document"]
sent_runs = [n for n in ORDER if RESULTS[n]["context"] == "sentence"]
for s in sent_runs:
    twin = [n for n in doc_runs if RESULTS[n]["method"] == RESULTS[s]["method"]]
    if twin:
        print(f"Context ablation ({RESULTS[s]['method']}): sentence-only {RESULTS[s]['dev']['all']['macro_f1']:.4f} vs "
              f"document context {RESULTS[twin[0]]['dev']['all']['macro_f1']:.4f}")
if "E4_distill" in RESULTS:
    r = RESULTS["E4_distill"]
    tn = [n for n in r["cfg"].get("teacher_name", "").split("+") if n in RESULTS]
    if tn:
        t0 = RESULTS[tn[0]]; tp = sum(RESULTS[n]["params_backbone"] for n in tn)
        print(f"KD: student {r['params_backbone']/1e6:.1f}M F1 {r['dev']['all']['macro_f1']:.4f} vs teacher(s) "
              f"{'+'.join(tn)} {tp/1e6:.1f}M F1 {t0['dev']['all']['macro_f1']:.4f}; "
              f"speed {r['sentences_per_s']/max(1e-9, t0['sentences_per_s']):.2f}x")
for n in ORDER:
    d = RESULTS[n]["dev"]
    if "CL" in d and "IT" in d:
        print(f"{n:28s}: CL {d['CL']['macro_f1']:.4f} | IT {d['IT']['macro_f1']:.4f}")

In [ ]:
def load_scores(name, split):
    z = np.load(OUTPUT_DIR / "runs" / name / f"{split}_scores.npz")
    return {k[4:]: z[k].astype(np.float32) for k in z.files if k.startswith("doc_")}, (z["__trans"], z["__start"], z["__end"])

def _lsm(a):
    m = a.max(-1, keepdims=True)
    return a - m - np.log(np.exp(a - m).sum(-1, keepdims=True))

def combine(members, split):
    L = [load_scores(n, split) for n in members]
    sc = {k: np.mean([_lsm(x[0][k]) for x in L], 0) for k in L[0][0]}
    return sc, tuple(np.mean([x[1][i] for x in L], 0) for i in range(3))

def ensemble_dev(members):
    sc, pack = combine(members, "dev")
    pred = {d["id"]: viterbi(sc[d["id"]], pack) for d in DATA["dev"]}
    g = np.concatenate([d["labels"] for d in DATA["dev"]]); p = np.concatenate([pred[d["id"]] for d in DATA["dev"]])
    return macro_scores(g, p)["macro_f1"], pred

In [ ]:
SUB_DIR = OUTPUT_DIR / "submissions"
SUB_DIR.mkdir(exist_ok=True)
print(f"{'experiment':30s} {'dev macro-F1':>12s}  zip")
for n in ORDER:
    zp = OUTPUT_DIR / "runs" / n / "submission.zip"
    if not zp.exists():
        continue
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"], z.namelist()
        sub = json.loads(z.read("prediction.json"))
    assert set(sub) == set(test_ids)
    for d in DATA["test"]:
        assert len(sub[d["id"]]) == len(d["sentences"])
    dest = SUB_DIR / f"phase3_{ARCH_NAME}_{n}.zip"
    shutil.copy(zp, dest)
    print(f"{n:30s} {RESULTS[n]['dev']['all']['macro_f1']:12.4f}  {dest}")
print("\nEach zip: prediction.json at the root, every CL+IT test id, one official label per sentence in order.")
print("Upload on Codabench (Phase 3). Course rule: your best four submissions must involve different Phase-1 models.")